# PM2.5 Prediction Model Development Process

This notebook contains the complete model development process, including:
- Data exploration
- Feature engineering
- Model selection
- Hyperparameter tuning
- Model evaluation

In [20]:
# Import necessary libraries
import numpy as np
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from catboost import CatBoostRegressor, Pool
import warnings
import optuna
import json
import os
warnings.filterwarnings('ignore')

# Set Polars display options
pl.Config.set_tbl_rows(20)

polars.config.Config

## 1. Data Exploration

First, examine the basic structure and statistical information of the dataset

In [2]:
# Read training data
df_train = pl.read_csv('data/df-train.csv')

print("="*70)
print("Training Dataset Basic Information")
print("="*70)
print(f"Data shape: {df_train.shape}")
print(f"Number of rows: {df_train.height}")
print(f"Number of columns: {df_train.width}")
print(f"\nColumn names:")
print(df_train.columns)
print(f"\nData types (after reading):")
print(df_train.dtypes)

# Check which columns were read as strings but should be numeric
print(f"\n{'='*70}")
print("Data Type Check")
print(f"{'='*70}")

# Columns that should be numeric (excluding datetime)
numeric_cols_expected = [col for col in df_train.columns if col != 'datetime']
string_cols = []
for col in numeric_cols_expected:
    if df_train[col].dtype == pl.Utf8:
        string_cols.append(col)

if string_cols:
    print(f"The following columns were read as string type and need conversion: {len(string_cols)} columns")
    print(f"Column names: {string_cols[:10]}...")  # Only show first 10
    print("\nNote: These columns may contain 'NA' strings, which need to be converted to None before converting to numeric type")
else:
    print("All numeric columns have been correctly read as numeric types")

print(f"\nFirst 5 rows of data:")
print(df_train.head())

Training Dataset Basic Information
Data shape: (12121, 25)
Number of rows: 12121
Number of columns: 25

Column names:
['datetime', 'station1_temperature', 'station1_humidity', 'station1_windSpeed', 'station1_windBearing', 'station1_precipIntensity', 'station1_AQI', 'station1_PM2.5', 'station1_PM10', 'station1_SO2', 'station1_NO2', 'station1_O3', 'station1_CO', 'station2_temperature', 'station2_humidity', 'station2_windSpeed', 'station2_windBearing', 'station2_precipIntensity', 'station2_AQI', 'station2_PM2.5', 'station2_PM10', 'station2_SO2', 'station2_NO2', 'station2_O3', 'station2_CO']

Data types (after reading):
[String, Float64, Float64, Float64, Int64, Float64, String, String, String, String, String, String, String, Float64, Float64, Float64, Int64, Float64, String, String, String, String, String, String, String]

Data Type Check
The following columns were read as string type and need conversion: 14 columns
Column names: ['station1_AQI', 'station1_PM2.5', 'station1_PM10', 'statio

In [3]:
# Read forecast data
df_forecast = pl.read_csv('data/df-forecast.csv')

print("="*70)
print("Forecast Dataset Basic Information")
print("="*70)
print(f"Data shape: {df_forecast.shape}")
print(f"Number of rows: {df_forecast.height}")
print(f"Number of columns: {df_forecast.width}")
print(f"\nColumn names:")
print(df_forecast.columns)
print(f"\nData types:")
print(df_forecast.dtypes)
print(f"\nFirst 5 rows of data:")
print(df_forecast.head())
print(f"\nLast 5 rows of data:")
print(df_forecast.tail())

Forecast Dataset Basic Information
Data shape: (24, 2)
Number of rows: 24
Number of columns: 2

Column names:
['datetime', 'station1_PM2.5']

Data types:
[String, Float64]

First 5 rows of data:
shape: (5, 2)
┌─────────────────────────────────┬────────────────┐
│ datetime                        ┆ station1_PM2.5 │
│ ---                             ┆ ---            │
│ str                             ┆ f64            │
╞═════════════════════════════════╪════════════════╡
│ 2022-12-19T01:00:00.000000+000… ┆ 8.25638        │
│ 2022-12-19T02:00:00.000000+000… ┆ 8.264928       │
│ 2022-12-19T03:00:00.000000+000… ┆ 8.199979       │
│ 2022-12-19T04:00:00.000000+000… ┆ 8.258198       │
│ 2022-12-19T05:00:00.000000+000… ┆ 8.296917       │
└─────────────────────────────────┴────────────────┘

Last 5 rows of data:
shape: (5, 2)
┌─────────────────────────────────┬────────────────┐
│ datetime                        ┆ station1_PM2.5 │
│ ---                             ┆ ---            │
│ str        

In [6]:
# Check missing values in training data (before type conversion)
print("="*70)
print("Training Data Missing Value Statistics (Raw Data)")
print("="*70)
null_counts = df_train.null_count()
print(null_counts)
print("\nNote: This statistics is before type conversion, some 'NA' strings have not yet been recognized as missing values")

Training Data Missing Value Statistics (Raw Data)
shape: (1, 25)
┌──────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ datetime ┆ station1_ ┆ station1_ ┆ station1_ ┆ … ┆ station2_ ┆ station2_ ┆ station2_ ┆ station2_ │
│ ---      ┆ temperatu ┆ humidity  ┆ windSpeed ┆   ┆ SO2       ┆ NO2       ┆ O3        ┆ CO        │
│ u32      ┆ re        ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│          ┆ ---       ┆ u32       ┆ u32       ┆   ┆ u32       ┆ u32       ┆ u32       ┆ u32       │
│          ┆ u32       ┆           ┆           ┆   ┆           ┆           ┆           ┆           │
╞══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0        ┆ 0         ┆ 0         ┆ 0         ┆ … ┆ 89        ┆ 89        ┆ 89        ┆ 89        │
└──────────┴───────────┴───────────┴───────────┴───┴───────────┴───────────┴───────────┴───────────┘

Note: This statistics is 

## 2. Data Cleaning and Type Conversion

Convert all numeric columns from string type to correct numeric types

In [5]:
# Data cleaning: convert types of all numeric columns
print("="*70)
print("Data Cleaning and Type Conversion")
print("="*70)

# Check and convert datetime column
if df_train['datetime'].dtype == pl.Utf8:
    df_train = df_train.with_columns(pl.col('datetime').str.to_datetime())
    print("✓ datetime column has been converted to datetime type")

# Convert all numeric columns (excluding datetime)
numeric_cols_to_convert = [col for col in df_train.columns if col != 'datetime' and df_train[col].dtype == pl.Utf8]

if numeric_cols_to_convert:
    print(f"\nFound {len(numeric_cols_to_convert)} numeric columns that need conversion")
    
    # Create conversion expressions for each column
    cast_exprs = []
    for col in numeric_cols_to_convert:
        # Convert 'NA' strings to None, then convert to Float64
        cast_exprs.append(
            pl.when(pl.col(col) == 'NA')
            .then(None)
            .otherwise(pl.col(col))
            .cast(pl.Float64, strict=False)
            .alias(col)
        )
    
    # Apply conversion
    df_train = df_train.with_columns(cast_exprs)
    print("✓ All numeric columns have been converted to Float64 type")
else:
    print("✓ All column types are correct, no conversion needed")

# Check data types after conversion
print(f"\nData types after conversion:")
print(df_train.dtypes[:10])  # Only show first 10
print("...")

# Check target variable statistics
target = 'station1_PM2.5'
print(f"\n{'='*70}")
print(f"Target Variable {target} Statistics")
print(f"{'='*70}")
target_stats = df_train.select([
    pl.col(target).mean().alias('mean'),
    pl.col(target).std().alias('std'),
    pl.col(target).min().alias('min'),
    pl.col(target).max().alias('max'),
    pl.col(target).quantile(0.25).alias('q25'),
    pl.col(target).quantile(0.5).alias('median'),
    pl.col(target).quantile(0.75).alias('q75'),
    pl.col(target).count().alias('count'),
    pl.col(target).null_count().alias('null_count')
])
print(target_stats)

Data Cleaning and Type Conversion
✓ datetime column has been converted to datetime type

Found 14 numeric columns that need conversion
✓ All numeric columns have been converted to Float64 type

Data types after conversion:
[Datetime(time_unit='us', time_zone='UTC'), Float64, Float64, Float64, Int64, Float64, Float64, Float64, Float64, Float64]
...

Target Variable station1_PM2.5 Statistics
shape: (1, 9)
┌───────────┬───────────┬─────┬───────┬───┬────────┬──────┬───────┬────────────┐
│ mean      ┆ std       ┆ min ┆ max   ┆ … ┆ median ┆ q75  ┆ count ┆ null_count │
│ ---       ┆ ---       ┆ --- ┆ ---   ┆   ┆ ---    ┆ ---  ┆ ---   ┆ ---        │
│ f64       ┆ f64       ┆ f64 ┆ f64   ┆   ┆ f64    ┆ f64  ┆ u32   ┆ u32        │
╞═══════════╪═══════════╪═════╪═══════╪═══╪════════╪══════╪═══════╪════════════╡
│ 15.928939 ┆ 11.826574 ┆ 1.0 ┆ 130.0 ┆ … ┆ 13.0   ┆ 22.0 ┆ 12032 ┆ 89         │
└───────────┴───────────┴─────┴───────┴───┴────────┴──────┴───────┴────────────┘


In [7]:
# Check datetime column
print("="*70)
print("Time Range Information")
print("="*70)

# Check the type of datetime column
print(f"datetime column type: {df_train['datetime'].dtype}")

# If datetime is a string, try to convert
if df_train['datetime'].dtype == pl.Utf8:
    df_train = df_train.with_columns(pl.col('datetime').str.to_datetime())
    print("datetime column has been converted to datetime type")

# Display time range
if df_train['datetime'].dtype in [pl.Datetime, pl.Datetime('us', 'UTC')]:
    print(f"Training data time range:")
    print(f"  Start time: {df_train['datetime'].min()}")
    print(f"  End time: {df_train['datetime'].max()}")
    print(f"  Time span: {(df_train['datetime'].max() - df_train['datetime'].min())}")
    
    # Check forecast data time range
    if df_forecast['datetime'].dtype == pl.Utf8:
        df_forecast = df_forecast.with_columns(pl.col('datetime').str.to_datetime())
    
    if df_forecast['datetime'].dtype in [pl.Datetime, pl.Datetime('us', 'UTC')]:
        print(f"\nForecast data time range:")
        print(f"  Start time: {df_forecast['datetime'].min()}")
        print(f"  End time: {df_forecast['datetime'].max()}")
        print(f"  Time span: {(df_forecast['datetime'].max() - df_forecast['datetime'].min())}")

Time Range Information
datetime column type: Datetime(time_unit='us', time_zone='UTC')
Training data time range:
  Start time: 2021-08-01 00:00:00+00:00
  End time: 2022-12-19 00:00:00+00:00
  Time span: 505 days, 0:00:00

Forecast data time range:
  Start time: 2022-12-19 01:00:00+00:00
  End time: 2022-12-20 00:00:00+00:00
  Time span: 23:00:00


## 3. Feature Engineering

Create time features, lag features, and rolling window features

In [9]:
# Define feature engineering functions
def create_time_features(df):
    """Create time-related features"""
    df = df.with_columns([
        pl.col('datetime').dt.hour().alias('hour'),
        pl.col('datetime').dt.day().alias('day'),
        pl.col('datetime').dt.month().alias('month'),
        pl.col('datetime').dt.weekday().alias('weekday'),
        # Cyclical encoding (sin/cos transformation)
        (np.sin(pl.col('datetime').dt.hour() * 2 * np.pi / 24)).alias('hour_sin'),
        (np.cos(pl.col('datetime').dt.hour() * 2 * np.pi / 24)).alias('hour_cos'),
        (np.sin(pl.col('datetime').dt.day() * 2 * np.pi / 31)).alias('day_sin'),
        (np.cos(pl.col('datetime').dt.day() * 2 * np.pi / 31)).alias('day_cos'),
        (np.sin(pl.col('datetime').dt.month() * 2 * np.pi / 12)).alias('month_sin'),
        (np.cos(pl.col('datetime').dt.month() * 2 * np.pi / 12)).alias('month_cos'),
        (np.sin(pl.col('datetime').dt.weekday() * 2 * np.pi / 7)).alias('weekday_sin'),
        (np.cos(pl.col('datetime').dt.weekday() * 2 * np.pi / 7)).alias('weekday_cos'),
    ])
    return df

def create_lag_features(df, target_col='station1_PM2.5', lags=[1, 2, 3, 6, 12, 24]):
    """Create lag features"""
    for lag in lags:
        df = df.with_columns(
            pl.col(target_col).shift(lag).alias(f'{target_col}_lag{lag}')
        )
    return df

def create_rolling_features(df, target_col='station1_PM2.5'):
    """Create rolling window features"""
    df = df.with_columns([
        # Rolling mean
        pl.col(target_col).rolling_mean(window_size=3).shift(1).alias(f'{target_col}_roll_mean_3'),
        pl.col(target_col).rolling_mean(window_size=6).shift(1).alias(f'{target_col}_roll_mean_6'),
        pl.col(target_col).rolling_mean(window_size=12).shift(1).alias(f'{target_col}_roll_mean_12'),
        pl.col(target_col).rolling_mean(window_size=24).shift(1).alias(f'{target_col}_roll_mean_24'),
        # Rolling standard deviation
        pl.col(target_col).rolling_std(window_size=3).shift(1).alias(f'{target_col}_roll_std_3'),
        pl.col(target_col).rolling_std(window_size=6).shift(1).alias(f'{target_col}_roll_std_6'),
        pl.col(target_col).rolling_std(window_size=12).shift(1).alias(f'{target_col}_roll_std_12'),
        pl.col(target_col).rolling_std(window_size=24).shift(1).alias(f'{target_col}_roll_std_24'),
        # Rolling maximum and minimum
        pl.col(target_col).rolling_max(window_size=24).shift(1).alias(f'{target_col}_roll_max_24'),
        pl.col(target_col).rolling_min(window_size=24).shift(1).alias(f'{target_col}_roll_min_24'),
    ])
    return df

print("Feature engineering functions defined")

Feature engineering functions defined


In [12]:
# Apply feature engineering to training data
print("="*70)
print("Applying Feature Engineering")
print("="*70)

# Apply feature engineering
df_train_fe = create_time_features(df_train)
df_train_fe = create_lag_features(df_train_fe, target_col='station1_PM2.5')
df_train_fe = create_rolling_features(df_train_fe, target_col='station1_PM2.5')

print(f"Data shape before feature engineering: {df_train.shape}")
print(f"Data shape after feature engineering: {df_train_fe.shape}")

# Define target variable and feature list
target = 'station1_PM2.5'
exclude_cols = ['datetime', target]
all_features = [col for col in df_train_fe.columns if col not in exclude_cols]

print(f"\nTotal number of features: {len(all_features)}")
print(f"  - Original features: {len([c for c in all_features if not any(x in c for x in ['lag', 'roll', 'sin', 'cos', 'hour', 'day', 'month', 'weekday'])])}")
print(f"  - Time features: {len([c for c in all_features if any(x in c for x in ['hour', 'day', 'month', 'weekday']) and not any(x in c for x in ['lag', 'roll'])])}")
print(f"  - Lag features: {len([c for c in all_features if 'lag' in c])}")
print(f"  - Rolling features: {len([c for c in all_features if 'roll' in c])}")

Applying Feature Engineering
Data shape before feature engineering: (12121, 25)
Data shape after feature engineering: (12121, 53)

Total number of features: 51
  - Original features: 23
  - Time features: 12
  - Lag features: 6
  - Rolling features: 10


## 4. Data Preparation and Splitting

Clean missing values and split training and validation sets

In [13]:
# Remove missing values
print("="*70)
print("Data Cleaning")
print("="*70)

df_train_clean = df_train_fe.drop_nulls()

print(f"Data shape before cleaning: {df_train_fe.shape}")
print(f"Data shape after cleaning: {df_train_clean.shape}")
print(f"Removed {df_train_fe.shape[0] - df_train_clean.shape[0]} rows ({(df_train_fe.shape[0] - df_train_clean.shape[0]) / df_train_fe.shape[0] * 100:.2f}%)")

# Time series data splitting (using 80-20 split, no shuffle)
split_idx = int(len(df_train_clean) * 0.8)
df_train_split = df_train_clean[:split_idx]
df_valid_split = df_train_clean[split_idx:]

print(f"\nData splitting:")
print(f"Training set size: {df_train_split.shape[0]} ({df_train_split.shape[0]/len(df_train_clean)*100:.1f}%)")
print(f"Validation set size: {df_valid_split.shape[0]} ({df_valid_split.shape[0]/len(df_train_clean)*100:.1f}%)")
print(f"Training set time range: {df_train_split['datetime'].min()} to {df_train_split['datetime'].max()}")
print(f"Validation set time range: {df_valid_split['datetime'].min()} to {df_valid_split['datetime'].max()}")

Data Cleaning
Data shape before cleaning: (12121, 53)
Data shape after cleaning: (11104, 53)
Removed 1017 rows (8.39%)

Data splitting:
Training set size: 8883 (80.0%)
Validation set size: 2221 (20.0%)
Training set time range: 2021-08-02 00:00:00+00:00 to 2022-09-14 08:00:00+00:00
Validation set time range: 2022-09-14 09:00:00+00:00 to 2022-12-19 00:00:00+00:00


## 5. Model Training and Evaluation

Test different models and feature combinations

In [15]:
# Prepare training and validation data
X_train = df_train_split.select(all_features).to_numpy()
y_train = df_train_split.get_column(target).to_numpy()
X_valid = df_valid_split.select(all_features).to_numpy()
y_valid = df_valid_split.get_column(target).to_numpy()


print("Data Preparation Complete")
print(f"Training set feature shape: {X_train.shape}")
print(f"Validation set feature shape: {X_valid.shape}")
print(f"Training set target shape: {y_train.shape}")
print(f"Validation set target shape: {y_valid.shape}")

Data Preparation Complete
Training set feature shape: (8883, 51)
Validation set feature shape: (2221, 51)
Training set target shape: (8883,)
Validation set target shape: (2221,)


In [17]:
# Test linear regression model (baseline model)

print("Testing Linear Regression Model (Baseline)")
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

y_pred_train_lr = lr_model.predict(X_train)
y_pred_valid_lr = lr_model.predict(X_valid)

mae_train_lr = mean_absolute_error(y_train, y_pred_train_lr)
mae_valid_lr = mean_absolute_error(y_valid, y_pred_valid_lr)
rmse_valid_lr = np.sqrt(mean_squared_error(y_valid, y_pred_valid_lr))

print(f"Linear Regression Results:")
print(f"  Training set MAE: {mae_train_lr:.4f}")
print(f"  Validation set MAE: {mae_valid_lr:.4f}")
print(f"  Validation set RMSE: {rmse_valid_lr:.4f}")

Testing Linear Regression Model (Baseline)
Linear Regression Results:
  Training set MAE: 1.5540
  Validation set MAE: 1.9463
  Validation set RMSE: 3.0353


In [18]:
# Test CatBoost model
print("="*70)
print("Testing CatBoost Model")
print("="*70)

# Prepare CatBoost Pool
train_pool = Pool(data=X_train, label=y_train, feature_names=all_features)
valid_pool = Pool(data=X_valid, label=y_valid, feature_names=all_features)
# Train CatBoost model
catboost_model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=6,
    loss_function='MAE',
    eval_metric='MAE',
    verbose=100,
    random_seed=42
)
catboost_model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=50)

# Predict
y_pred_train_cb = catboost_model.predict(X_train)
y_pred_valid_cb = catboost_model.predict(X_valid)

# Evaluate
mae_train_cb = mean_absolute_error(y_train, y_pred_train_cb)
mae_valid_cb = mean_absolute_error(y_valid, y_pred_valid_cb)
rmse_valid_cb = np.sqrt(mean_squared_error(y_valid, y_pred_valid_cb))

print(f"\nCatBoost Results:")
print(f"  Training set MAE: {mae_train_cb:.4f}")
print(f"  Validation set MAE: {mae_valid_cb:.4f}")
print(f"  Validation set RMSE: {rmse_valid_cb:.4f}")
print(f"\nModel Comparison:")
print(f"  Linear Regression Validation MAE: {mae_valid_lr:.4f}")
print(f"  CatBoost Validation MAE: {mae_valid_cb:.4f}")
print(f"  Improvement: {(mae_valid_lr - mae_valid_cb) / mae_valid_lr * 100:.2f}%")

Testing CatBoost Model
0:	learn: 8.3351674	test: 8.1366039	best: 8.1366039 (0)	total: 146ms	remaining: 1m 12s
100:	learn: 1.3038469	test: 1.7787290	best: 1.7787290 (100)	total: 448ms	remaining: 1.77s
200:	learn: 1.0669330	test: 1.6949352	best: 1.6941734 (199)	total: 731ms	remaining: 1.09s
300:	learn: 0.9520247	test: 1.6674963	best: 1.6668813 (299)	total: 1s	remaining: 661ms
400:	learn: 0.8831680	test: 1.6600290	best: 1.6600290 (400)	total: 1.27s	remaining: 313ms
499:	learn: 0.8242539	test: 1.6545713	best: 1.6543136 (498)	total: 1.55s	remaining: 0us

bestTest = 1.654313641
bestIteration = 498

Shrink model to first 499 iterations.

CatBoost Results:
  Training set MAE: 0.8246
  Validation set MAE: 1.6543
  Validation set RMSE: 2.7477

Model Comparison:
  Linear Regression Validation MAE: 1.9463
  CatBoost Validation MAE: 1.6543
  Improvement: 15.00%


## 6. Hyperparameter Tuning

Use Optuna for Bayesian optimization hyperparameter tuning

In [21]:
# Define hyperparameter tuning function
def auto_hyperparameter_tuning(X_train, y_train, X_valid, y_valid, 
                                feature_names, n_trials=100, 
                                study_name='catboost_pm25', 
                                storage_path='optuna_results/optuna_study.db'):   
    # Ensure storage directory exists
    os.makedirs(os.path.dirname(storage_path), exist_ok=True)
    
    # Prepare CatBoost Pool (defined outside objective function to ensure available in closure)
    train_pool = Pool(data=X_train, label=y_train, feature_names=feature_names)
    valid_pool = Pool(data=X_valid, label=y_valid, feature_names=feature_names)
    
    # Define objective function
    def objective(trial):
        # Hyperparameter search space
        params = {
            'iterations': trial.suggest_int('iterations', 300, 1000),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'depth': trial.suggest_int('depth', 4, 10),
            'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1, 10, log=True),
            'random_strength': trial.suggest_float('random_strength', 0, 1),
            'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
            'loss_function': 'MAE',
            'eval_metric': 'MAE',
            'random_seed': 42,
            'verbose': False
        }
        
        # Train model
        model = CatBoostRegressor(**params)
        model.fit(train_pool, eval_set=valid_pool, early_stopping_rounds=50, verbose=False)
        
        # Predict and calculate MAE
        y_pred = model.predict(X_valid)
        mae = mean_absolute_error(y_valid, y_pred)
        
        return mae
    
    # Create or load study
    try:
        study = optuna.load_study(study_name=study_name, storage=f'sqlite:///{storage_path}')
        print(f"Loaded existing study: {study_name}")
        print(f"Existing number of trials: {len(study.trials)}")
    except:
        study = optuna.create_study(
            study_name=study_name,
            storage=f'sqlite:///{storage_path}',
            direction='minimize',
            load_if_exists=True
        )
        print(f"Created new study: {study_name}")
    
    # Run optimization
    print(f"\nStarting hyperparameter optimization, target number of trials: {n_trials}")
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    # Save best parameters (round values to 3 decimal places)
    best_params_raw = study.best_params
    def _round_val(v):
        if isinstance(v, (int, np.integer)):
            return int(v)
        if isinstance(v, (float, np.floating)):
            return round(float(v), 3)
        return v
    best_params = {k: _round_val(v) for k, v in best_params_raw.items()}
    best_value = round(float(study.best_value), 4)
    
    # Save to JSON file
    results_dir = 'optuna_results'
    os.makedirs(results_dir, exist_ok=True)
    best_params_file = os.path.join(results_dir, 'best_params.json')
    
    with open(best_params_file, 'w', encoding='utf-8') as f:
        json.dump(best_params, f, indent=2, ensure_ascii=False)
    print(f"\n{'='*70}")
    print("Hyperparameter Optimization Complete")
    print(f"{'='*70}")
    print(f"Best parameters: {best_params}")
    print(f"Best MAE: {best_value:.4f}")
    print(f"Total number of trials: {len(study.trials)}")
    print(f"Best parameters saved to: {best_params_file}")
    
    return study, best_params, best_value

print("Hyperparameter tuning function defined")

Hyperparameter tuning function defined


In [22]:
# Run hyperparameter tuning
print("="*70)
print("Starting Hyperparameter Tuning")
print("="*70)

study, best_params, best_mae = auto_hyperparameter_tuning(
    X_train, y_train, X_valid, y_valid,
    feature_names=all_features,
    n_trials=100,  # Can adjust the number of trials as needed
    study_name='catboost_pm25',
    storage_path='optuna_results/optuna_study.db'
)

Starting Hyperparameter Tuning
Loaded existing study: catboost_pm25


  0%|          | 0/100 [00:00<?, ?it/s]

Existing number of trials: 2400

Starting hyperparameter optimization, target number of trials: 100


Best trial: 1980. Best value: 1.52701:   1%|          | 1/100 [00:01<01:43,  1.05s/it]

[I 2025-12-12 12:58:24,593] Trial 2400 finished with value: 1.6164040101953976 and parameters: {'iterations': 989, 'learning_rate': 0.07531827555143321, 'depth': 4, 'l2_leaf_reg': 3.215486974505388, 'random_strength': 0.4751692622470132, 'bagging_temperature': 0.494774255942731}. Best is trial 1980 with value: 1.5270065646838191.


...

[I 2025-12-12 13:01:26,227] Trial 2497 finished with value: 1.5900437355281265 and parameters: {'iterations': 939, 'learning_rate': 0.0631030239481646, 'depth': 4, 'l2_leaf_reg': 2.0390625355806278, 'random_strength': 0.4089020158982932, 'bagging_temperature': 0.3605064093455725}. Best is trial 1980 with value: 1.5270065646838191.


Best trial: 1980. Best value: 1.52701:  99%|█████████▉| 99/100 [03:03<00:01,  1.56s/it]

[I 2025-12-12 13:01:27,507] Trial 2498 finished with value: 1.5888760115195593 and parameters: {'iterations': 571, 'learning_rate': 0.07986022555571078, 'depth': 4, 'l2_leaf_reg': 1.3252469557176043, 'random_strength': 0.31807059037624985, 'bagging_temperature': 0.018865645298690073}. Best is trial 1980 with value: 1.5270065646838191.


Best trial: 1980. Best value: 1.52701: 100%|██████████| 100/100 [03:05<00:00,  1.86s/it]


[I 2025-12-12 13:01:29,277] Trial 2499 finished with value: 1.6268089506310621 and parameters: {'iterations': 989, 'learning_rate': 0.054924836537399664, 'depth': 5, 'l2_leaf_reg': 1.5524686923941386, 'random_strength': 0.2967465545577321, 'bagging_temperature': 0.5293135671755791}. Best is trial 1980 with value: 1.5270065646838191.

Hyperparameter Optimization Complete
Best parameters: {'iterations': 938, 'learning_rate': 0.061, 'depth': 4, 'l2_leaf_reg': 3.311, 'random_strength': 0.376, 'bagging_temperature': 0.408}
Best MAE: 1.5270
Total number of trials: 2500
Best parameters saved to: optuna_results\best_params.json


## 7. Train Final Model with Optimized Hyperparameters

In [23]:
# Train model with optimized hyperparameters
print("="*70)
print("Training Model with Optuna Optimized Hyperparameters")
print("="*70)

# Try to load best parameters (from variable or file)
try:
    # First try to use defined variable
    if 'best_params' in globals():
        best_params_loaded = best_params.copy()
        print("✓ Using hyperparameter tuning results that have been run")
    else:
        # If variable doesn't exist, try to load from file
        import json
        best_params_file = 'optuna_results/best_params.json'
        with open(best_params_file, 'r', encoding='utf-8') as f:
            best_params_loaded = json.load(f)
        print(f"✓ Loaded best parameters from file: {best_params_file}")
except FileNotFoundError:
    # If file also doesn't exist, use default parameters
    print("⚠ Optimized parameters not found, using default parameters")
    best_params_loaded = {
        'iterations': 500,
        'learning_rate': 0.1,
        'depth': 6
    }
except Exception as e:
    print(f"⚠ Error loading parameters: {e}, using default parameters")
    best_params_loaded = {
        'iterations': 500,
        'learning_rate': 0.1,
        'depth': 6
    }

# Prepare best parameters (add necessary parameters)
best_model_params = best_params_loaded.copy()
best_model_params.update({
    'loss_function': 'MAE',
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': 100
})

print(f"Best parameters: {best_model_params}")

# Train final model
train_pool_final = Pool(data=X_train, label=y_train, feature_names=all_features)
valid_pool_final = Pool(data=X_valid, label=y_valid, feature_names=all_features)

final_model = CatBoostRegressor(**best_model_params)
final_model.fit(
    train_pool_final, 
    eval_set=valid_pool_final, 
    early_stopping_rounds=50
)

# Predict
y_pred_train_final = final_model.predict(X_train)
y_pred_valid_final = final_model.predict(X_valid)

# Evaluate
mae_train_final = mean_absolute_error(y_train, y_pred_train_final)
mae_valid_final = mean_absolute_error(y_valid, y_pred_valid_final)
rmse_valid_final = np.sqrt(mean_squared_error(y_valid, y_pred_valid_final))

print(f"\n{'='*70}")
print("Optuna Optimized Model Results:")
print(f"{'='*70}")
print(f"Training set MAE: {mae_train_final:.4f}")
print(f"Validation set MAE: {mae_valid_final:.4f}")
print(f"Validation set RMSE: {rmse_valid_final:.4f}")

print(f"\n{'='*70}")
print("Model Comparison:")
print(f"{'='*70}")
print(f"{'Model':<25} {'Validation MAE':<15} {'Validation RMSE':<15}")
print("-" * 70)
print(f"{'CatBoost (Default)':<25} {mae_valid_cb:<15.4f} {rmse_valid_cb:<15.4f}")

# Try to display best MAE during optimization (if exists)
try:
    if 'best_mae' in globals():
        print(f"{'Optuna Best During Tuning':<25} {best_mae:<15.4f} {'-':<15}")
except:
    pass

print(f"{'Optuna Optimized':<25} {mae_valid_final:<15.4f} {rmse_valid_final:<15.4f}")

improvement = (mae_valid_cb - mae_valid_final) / mae_valid_cb * 100
print(f"\nImprovement over default parameters: {improvement:.2f}%")

print(f"\n{'='*70}")
print("✓ Optimized model training complete!")
print(f"{'='*70}")

Training Model with Optuna Optimized Hyperparameters
✓ Using hyperparameter tuning results that have been run
Best parameters: {'iterations': 938, 'learning_rate': 0.061, 'depth': 4, 'l2_leaf_reg': 3.311, 'random_strength': 0.376, 'bagging_temperature': 0.408, 'loss_function': 'MAE', 'eval_metric': 'MAE', 'random_seed': 42, 'verbose': 100}
0:	learn: 8.6318391	test: 8.4315741	best: 8.4315741 (0)	total: 2.34ms	remaining: 2.19s
100:	learn: 1.4880756	test: 1.7749034	best: 1.7749034 (100)	total: 172ms	remaining: 1.42s
200:	learn: 1.3305395	test: 1.6749490	best: 1.6749490 (200)	total: 390ms	remaining: 1.43s
300:	learn: 1.2368319	test: 1.6310499	best: 1.6307725 (298)	total: 561ms	remaining: 1.19s
400:	learn: 1.1779083	test: 1.6165633	best: 1.6160423 (389)	total: 736ms	remaining: 985ms
500:	learn: 1.1347801	test: 1.6047469	best: 1.6040075 (498)	total: 902ms	remaining: 786ms
600:	learn: 1.0976914	test: 1.5919147	best: 1.5919147 (600)	total: 1.09s	remaining: 611ms
700:	learn: 1.0668416	test: 1.5

In [24]:
print("Validation set MAE:", mae_valid_final)
print("Validation set RMSE:", rmse_valid_final)
print("Best parameters:", best_model_params)

Validation set MAE: 1.5698410646376384
Validation set RMSE: 2.662769208792807
Best parameters: {'iterations': 938, 'learning_rate': 0.061, 'depth': 4, 'l2_leaf_reg': 3.311, 'random_strength': 0.376, 'bagging_temperature': 0.408, 'loss_function': 'MAE', 'eval_metric': 'MAE', 'random_seed': 42, 'verbose': 100}
